In [0]:
%reload_ext autoreload
%autoreload 2
spark.conf.set("spark.sql.session.timeZone", "Asia/Kolkata")

df_sales = spark.read.table("pyspark_real_time.bronze.sales")
df_sales.show(5,truncate=False)

In [0]:
df_prod = spark.read.table("pyspark_real_time.bronze.products")
df_prod.show(5,truncate=False)

In [0]:
## dynamic silver transformations
from utils.custom_utils import Transformations     

# Initialize transformation class once
tf = Transformations(spark)

# Entity configuration dictionary
# Each entity defines:
# - keys → Primary key columns for dedup & merge
# - cdc  → CDC column for incremental updates
entities = {
    "sales": {"keys": ["sale_id"], "cdc": "date"}}

# Loop through each entity
for entity, config in entities.items():

    # Print section header for clarity
    print(f"Processing Entity: {entity}")
 

    # Read Bronze table
    print(f"Reading bronze table: pyspark_real_time.bronze.{entity}")
    df = spark.read.table(f"pyspark_real_time.bronze.{entity}")

    # Apply deduplication logic
    print("Applying deduplication...")
    df_tf = tf.dedup(df, config["keys"], config["cdc"])

    # # Apply process timestamp transformation
    # print("Applying process timestamp...")
    # df_tf = tf.process_timestamp(df_tf)

    # Define silver table name
    silver_table = f"pyspark_real_time.silver.{entity}"

    # Check if silver table exists
    if spark.catalog.tableExists(silver_table):

        # Perform upsert if table already exists
        print(f"Silver table exists: {silver_table}")
        print("Performing UPSERT...")

        tf.upsert(
            df_tf,
            entity,
            config["keys"],
            config["cdc"]
        )

        print(f"Upsert completed for {entity}")

    else:

        # Create table if it does not exist
        print(f"Silver table does not exist: {silver_table}")
        print("Creating new Delta table...")

        df_tf.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(silver_table)

        print(f"Table created: {silver_table}")

    # Completion message
    print(f"Completed processing for {entity}")

In [0]:
%sql
drop table pyspark_real_time.silver.sales;